In [21]:
from pathlib import Path
import json


def find_hierarchy_path() -> Path:
    cwd = Path.cwd()
    for candidate in (
        cwd / "output" / "hierarchy_lgd.json",
        cwd / "LGD" / "output" / "hierarchy_lgd.json",
    ):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find hierarchy_lgd.json. Run from repo root (Dhara) or LGD/."
    )


path = find_hierarchy_path()
with path.open(encoding="utf-8") as f:
    hierarchy_data = json.load(f)

print(f"Loaded from {path}")
print("Expected counts:", hierarchy_data["validation"]["counts"])

Loaded from /home/ubuntu/data-dhara/Dhara/LGD/output/hierarchy_lgd.json
Expected counts: {'state': 1, 'district': 13, 'sub_district': 39, 'village': 358, 'local_body': 4, 'cantonment_board': 1, 'ward': 280}


In [22]:
import re
from collections import Counter

TOKEN_RE = re.compile(r"[0-9a-z]+")


def normalize_name(name: str) -> str:
    return re.sub(r"\s+", " ", str(name).strip().casefold())


def tokenize(name: str) -> tuple[str, ...]:
    return tuple(TOKEN_RE.findall(normalize_name(name)))


def compact_name(name: str) -> str:
    return "".join(tokenize(name))


def scalar_text(value) -> str | None:
    if isinstance(value, str):
        text = value.strip()
        return text or None
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    return None


def add_to_bucket(index: dict, key: str, record: dict) -> None:
    if not key:
        return
    bucket = index.setdefault(key, [])
    if all(existing is not record for existing in bucket):
        bucket.append(record)


def humanize(label: str) -> str:
    return " ".join(part.capitalize() for part in str(label).split("_") if part)


def display_name(node: dict) -> str:
    entity_type = str(node.get("entity_type") or "")
    named = []
    for key, value in node.items():
        text = scalar_text(value)
        if text and "name" in key:
            named.append((key, text))
    preferred = [text for key, text in named if entity_type and entity_type in key and "english" in key]
    if not preferred:
        preferred = [text for key, text in named if entity_type and entity_type in key]
    if not preferred:
        preferred = [text for key, text in named if "english" in key]
    if not preferred and named:
        preferred = [named[0][1]]
    return preferred[0] if preferred else entity_type


def detect_hierarchy_separator(root) -> str:
    def walk_for_separator(node):
        if isinstance(node, dict):
            hierarchy = node.get("hierarchy")
            if isinstance(hierarchy, str):
                match = re.search(r"\)([^0-9A-Za-z]+)(?=\S)", hierarchy)
                if match:
                    return match.group(1)
            for value in node.values():
                found = walk_for_separator(value)
                if found:
                    return found
        elif isinstance(node, list):
            for item in node:
                found = walk_for_separator(item)
                if found:
                    return found
        return None

    return walk_for_separator(root) or " "


def ancestor_names(ancestors: list[dict]) -> set[str]:
    names = set()
    for ancestor in ancestors:
        for key, value in ancestor.items():
            text = scalar_text(value)
            if text and "name" in key:
                names.add(normalize_name(text))
    return names


def own_alias_texts(node: dict, ancestors: list[dict]) -> list[str]:
    entity_type = str(node.get("entity_type") or "")
    taken = ancestor_names(ancestors)
    texts = []
    seen = set()
    for key, value in node.items():
        text = scalar_text(value)
        if not text or ("name" not in key and "number" not in key):
            continue
        copied = entity_type not in key and normalize_name(text) in taken
        if copied:
            continue
        folded = normalize_name(text)
        if folded in seen:
            continue
        seen.add(folded)
        texts.append(text)
    return texts


def own_codes(node: dict, ancestors: list[dict]) -> list[tuple[str, str]]:
    inherited: dict[str, set[str]] = {}
    for ancestor in ancestors:
        for key, value in ancestor.items():
            text = scalar_text(value)
            if text and key.endswith("_code"):
                inherited.setdefault(key, set()).add(text)
    pairs = []
    for key, value in node.items():
        text = scalar_text(value)
        if not text or not key.endswith("_code"):
            continue
        if text in inherited.get(key, set()):
            continue
        pairs.append((key, text))
    return pairs


def primary_code(pairs: list[tuple[str, str]], entity_type: str) -> str | None:
    if not pairs:
        return None
    chosen = sorted(
        pairs,
        key=lambda item: (
            not ("lgd" in item[0] and entity_type in item[0]),
            entity_type not in item[0],
            "lgd" not in item[0],
            item[0],
        ),
    )
    return chosen[0][1]


def index_entity(node: dict, ancestors: list[dict]) -> dict:
    entity_type = str(node["entity_type"])
    display = display_name(node)
    alias_texts = own_alias_texts(node, ancestors)
    if display not in alias_texts and normalize_name(display) not in {normalize_name(text) for text in alias_texts}:
        alias_texts.insert(0, display)

    code_pairs = own_codes(node, ancestors)
    code = primary_code(code_pairs, entity_type)

    source_hierarchy = node.get("hierarchy")
    if isinstance(source_hierarchy, str) and source_hierarchy.strip():
        hierarchy = source_hierarchy.strip()
    else:
        labels = [
            f"{display_name(ancestor)}({humanize(ancestor['entity_type'])})"
            for ancestor in reversed(ancestors)
        ]
        hierarchy = hierarchy_separator.join(labels) if labels else None

    source_parts = node.get("hierarchy_parts")
    if isinstance(source_parts, dict) and source_parts:
        hierarchy_parts = dict(source_parts)
    elif ancestors:
        hierarchy_parts = {
            str(ancestor["entity_type"]): display_name(ancestor) for ancestor in ancestors
        }
    else:
        hierarchy_parts = None

    folded_display = normalize_name(display)
    aliases = [text for text in alias_texts if normalize_name(text) != folded_display]

    record = {
        "entity_type": entity_type,
        "lgd_code": code,
        "name_english": display,
        "aliases": aliases,
        "hierarchy": hierarchy,
        "hierarchy_parts": hierarchy_parts,
    }

    for text in alias_texts:
        add_to_bucket(lookup, normalize_name(text), record)
        add_to_bucket(compact_lookup, compact_name(text), record)
        for token in tokenize(text):
            add_to_bucket(token_index, token, record)

    for _, coded in code_pairs:
        add_to_bucket(code_lookup, coded, record)
        add_to_bucket(code_lookup, coded.casefold(), record)

    record_by_id[id(record)] = record
    name_token_count[id(record)] = len(tokenize(display))
    entities.append(record)
    return record


def walk(node, ancestors: list[dict] | None = None) -> None:
    ancestors = ancestors or []
    if isinstance(node, dict) and isinstance(node.get("entity_type"), str):
        index_entity(node, ancestors)
        next_ancestors = ancestors + [node]
        for value in node.values():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and isinstance(item.get("entity_type"), str):
                        walk(item, next_ancestors)
        return
    if isinstance(node, list):
        for item in node:
            walk(item, ancestors)
        return
    if isinstance(node, dict):
        for value in node.values():
            if isinstance(value, (dict, list)):
                walk(value, ancestors)


lookup: dict[str, list[dict]] = {}
compact_lookup: dict[str, list[dict]] = {}
code_lookup: dict[str, list[dict]] = {}
token_index: dict[str, list[dict]] = {}
record_by_id: dict[int, dict] = {}
name_token_count: dict[int, int] = {}
entities: list[dict] = []
hierarchy_separator = detect_hierarchy_separator(hierarchy_data)

walk(hierarchy_data)

counts = Counter(record["entity_type"] for record in entities)
print(
    f"Indexed {len(entities)} entities, {len(lookup)} names, "
    f"{len(token_index)} tokens, {len(code_lookup)} codes"
)
for entity_type, count in counts.most_common():
    print(f"  {entity_type}: {count}")

expected = {}
validation = hierarchy_data.get("validation")
if isinstance(validation, dict) and isinstance(validation.get("counts"), dict):
    expected = validation["counts"]
if expected:
    mismatches = {
        entity_type: {"expected": expected[entity_type], "indexed": counts.get(entity_type, 0)}
        for entity_type in set(expected) | set(counts)
        if expected.get(entity_type) != counts.get(entity_type)
    }
    print("Count check:", "ok" if not mismatches else mismatches)

Indexed 696 entities, 924 names, 732 tokens, 1141 codes
  village: 358
  ward: 280
  sub_district: 39
  district: 13
  local_body: 4
  state: 1
  cantonment_board: 1
Count check: ok


In [23]:
def levenshtein(left: str, right: str, budget: int) -> int | None:
    if left == right:
        return 0
    if budget <= 0 or abs(len(left) - len(right)) > budget:
        return None
    previous = list(range(len(right) + 1))
    for i, left_char in enumerate(left, 1):
        current = [i]
        row_min = i
        for j, right_char in enumerate(right, 1):
            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j - 1] + (left_char != right_char)
            value = min(insert_cost, delete_cost, replace_cost)
            current.append(value)
            if value < row_min:
                row_min = value
        if row_min > budget:
            return None
        previous = current
    distance = previous[-1]
    return distance if distance <= budget else None


def format_match(match: dict) -> str:
    lines = [
        f"{match['entity_type']}: {match['name_english']} → {match['lgd_code']}"
    ]
    display_key = normalize_name(str(match["name_english"]))
    for alias in match.get("aliases") or []:
        if normalize_name(str(alias)) != display_key:
            lines.append(f"  also: {alias}")
    if match.get("hierarchy"):
        lines.append(f"  hierarchy: {match['hierarchy']}")
    return "\n".join(lines)


def costs_for_token(query_token: str, fuzzy: bool) -> dict[int, int]:
    options: dict[int, int] = {}

    def offer(record: dict, cost: int) -> None:
        record_id = id(record)
        current = options.get(record_id)
        if current is None or cost < current:
            options[record_id] = cost

    for record in token_index.get(query_token, []):
        offer(record, 0)

    for token, records in token_index.items():
        if token == query_token or not token.startswith(query_token):
            continue
        gap = len(token) - len(query_token)
        for record in records:
            offer(record, gap)

    if fuzzy:
        budget = len(query_token) // 4
        if budget:
            for token, records in token_index.items():
                if token.startswith(query_token):
                    continue
                distance = levenshtein(query_token, token, budget)
                if not distance:
                    continue
                for record in records:
                    offer(record, distance)
    return options


def match_tokens(query_tokens: tuple[str, ...], fuzzy: bool) -> list[dict]:
    combined: dict[int, int] | None = None
    for query_token in query_tokens:
        options = costs_for_token(query_token, fuzzy)
        if not options:
            return []
        if combined is None:
            combined = options
            continue
        shared = {
            record_id: combined[record_id] + cost
            for record_id, cost in options.items()
            if record_id in combined
        }
        if not shared:
            return []
        combined = shared

    ranked = []
    for record_id, cost in (combined or {}).items():
        record = record_by_id[record_id]
        extra = max(0, name_token_count.get(record_id, 0) - len(query_tokens))
        ranked.append((cost, extra, record["name_english"] or "", record["lgd_code"] or "", record))
    ranked.sort()
    return [item[-1] for item in ranked]


def similar_records(query: str) -> list[dict]:
    """Whole-name edit distance, using one change per four characters of the query."""
    folded = compact_name(query)
    budget = len(folded) // 4
    if budget <= 0:
        return []

    ranked = []
    for record in entities:
        best = None
        texts = [record.get("name_english"), *(record.get("aliases") or [])]
        for text in texts:
            if not text:
                continue
            distance = levenshtein(folded, compact_name(text), budget)
            if distance is None:
                continue
            if best is None or distance < best:
                best = distance
        if best is None:
            continue
        ranked.append(
            (
                best,
                abs(len(compact_name(record.get("name_english") or "")) - len(folded)),
                record.get("name_english") or "",
                record.get("lgd_code") or "",
                record,
            )
        )
    ranked.sort()
    return [item[-1] for item in ranked]


def show_matches(matches: list[dict], kind: str, limit: int) -> list[dict]:
    shown = matches[:limit]
    print(f"{len(shown)} match(es) by {kind}:")
    for match in shown:
        print(format_match(match))
        print()
    if len(matches) > limit:
        print(f"Showing first {limit} of {len(matches)}. Narrow the name to see more.")
    return shown


def search(query: str, limit: int = 30, suggestions: int = 3) -> list[dict]:
    """Resolve a name or code.

    Tries an exact code, then an exact name, then the same name with spaces and
    punctuation removed, then token prefixes in any order, then per-token edits.
    Edit budget grows with the token: one change for every four characters.
    If those find nothing, returns the closest whole-name matches within that
    same budget, capped by ``suggestions``.
    """
    raw = str(query).strip()
    folded = normalize_name(raw)
    if not folded:
        print("Enter a name or code.")
        return []

    code_hits = []
    for key in (raw, folded):
        for record in code_lookup.get(key, []):
            if all(existing is not record for existing in code_hits):
                code_hits.append(record)
    if code_hits:
        return show_matches(code_hits, "code", limit)

    exact = lookup.get(folded, [])
    if exact:
        return show_matches(exact, "exact name", limit)

    compact_hits = compact_lookup.get(compact_name(raw), [])
    if compact_hits:
        return show_matches(compact_hits, "normalized name", limit)

    query_tokens = tokenize(raw)
    if not query_tokens:
        print(f"No match for {raw!r}.")
        return []

    partial = match_tokens(query_tokens, fuzzy=False)
    if partial:
        return show_matches(partial, "partial name", limit)

    fuzzy = match_tokens(query_tokens, fuzzy=True)
    if fuzzy:
        return show_matches(fuzzy, "fuzzy name", limit)

    similar = similar_records(raw)
    if similar:
        return show_matches(similar, "similar name", suggestions)

    print(f"No match for {raw!r}.")
    return []

In [24]:
# Re-run this cell and enter a name or code from the loaded hierarchy.
search(input("Enter a name or code: "))

3 match(es) by partial name:
ward: Bijwasan → 1290001
  also: 048-S
  hierarchy: South Delhi Municipal Corporation(Local Body)

sub_district: Bijwasan → 7580
  hierarchy: South West(District) / Delhi(State)

village: Bijwasan → 961372
  hierarchy: Bijwasan(Sub-District) / South West(District) / Delhi(State)



[{'entity_type': 'ward',
  'lgd_code': '1290001',
  'name_english': 'Bijwasan',
  'aliases': ['048-S'],
  'hierarchy': 'South Delhi Municipal Corporation(Local Body)',
  'hierarchy_parts': {'local_body': 'South Delhi Municipal Corporation'}},
 {'entity_type': 'sub_district',
  'lgd_code': '7580',
  'name_english': 'Bijwasan',
  'aliases': [],
  'hierarchy': 'South West(District) / Delhi(State)',
  'hierarchy_parts': {'district': 'South West', 'state': 'Delhi'}},
 {'entity_type': 'village',
  'lgd_code': '961372',
  'name_english': 'Bijwasan',
  'aliases': [],
  'hierarchy': 'Bijwasan(Sub-District) / South West(District) / Delhi(State)',
  'hierarchy_parts': {'sub_district': 'Bijwasan',
   'district': 'South West',
   'state': 'Delhi'}}]